# Lab 1: Your First Agent — Hello World
---
**SmartClaims AI Agent** | Microsoft Foundry Agent Service (v2.x)

## Objective
Create your first AI agent using the Foundry Agent Service. You'll learn:
- How to define an agent with custom instructions (system prompt)
- How to register agent versions in your Foundry project
- Single-turn request/response pattern
- Multi-turn conversations with context preservation
- Agent lifecycle management (create → use → delete)

## Architecture
```
User Input → OpenAI Client → Agent Reference → Foundry Agent (with instructions) → Model → Response
```

## Step 1: Import Dependencies

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from utils.config import get_clients, MODEL, print_header, print_step
from azure.ai.projects.models import PromptAgentDefinition

## Step 2: Initialize Clients
Create the Azure AI Project client and OpenAI client.

In [ ]:
project_client, openai_client = get_clients()
print("✅ Clients created")

## Step 3: Create an Agent Version

A **`PromptAgentDefinition`** defines the agent's:
- **Model** — Which deployed model to use
- **Instructions** — The system prompt that shapes the agent's personality and behavior

`create_version()` registers this definition in your Foundry project. Each call creates a new version, enabling A/B testing and safe rollbacks.

> 💡 **Key Concept:** Agents in Foundry are *versioned*. You can have multiple versions of the same agent running simultaneously, routing different traffic to each.

In [ ]:
agent = project_client.agents.create_version(
    agent_name="smartclaims-hello",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are SmartClaims, a friendly insurance assistant "
            "created by Contoso Insurance. You help customers "
            "with their insurance queries. Always be professional, "
            "clear, and helpful. When greeting someone, introduce "
            "yourself as SmartClaims."
            "DO NOT ANSWER anything apart from this role. Just Say I DONT KNOW"
        ),
    ),
)
print(f"Agent: {agent.name} (version: {agent.version})")

## Step 4: Send a Single Request

The `agent_reference` in `extra_body` routes the request to YOUR agent (with its custom instructions) rather than using the raw model directly.

This is the simplest interaction pattern — a single question with a single response.

In [ ]:
response = openai_client.responses.create(
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "version": agent.version,
            "type": "agent_reference",
        }
    },
    input=(
        "Hi! I just purchased a new auto insurance policy. "
        "What does it typically cover?"
    ),
)
print(f"👤 User: What does auto insurance cover?")
print(f"🤖 SmartClaims:\n{response.output_text}")

## Step 5: Multi-Turn Conversation

Real customer interactions involve multiple exchanges. The **Conversations API** maintains context across turns:

1. `conversations.create()` starts a stateful session
2. Passing `conversation=id` in each call links messages together
3. The agent automatically remembers previous context

> 💡 **Key Concept:** Without conversations, each call is independent. With a conversation ID, the agent sees the full history and can reference earlier messages.

In [ ]:
conversation = openai_client.conversations.create()

print(f"Conversation started (id: {conversation.id})")

def ask(user_input):
    """Helper to send messages within the same conversation."""
    return openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "version": agent.version,
                "type": "agent_reference",
            }
        },
        input=user_input,
    )

In [ ]:
# Turn 1
r1 = ask("I want to file a claim for a car accident that happened yesterday.")
print(f"👤 Turn 1: File a claim for car accident")
print(f"🤖 Response: {r1.output_text}")

In [ ]:
# Turn 2 — agent remembers the car accident context
r2 = ask("What documents will I need to submit?")
print(f"👤 Turn 2: What documents do I need?")
print(f"🤖 Response: {r2.output_text}")

In [ ]:
# Turn 2 — agent remembers the car accident context
r3 = ask("who is jalal?")

r3.output_text

> 📝 **Notice:** In Turn 2, the agent knows you're asking about documents for a *car accident claim* — even though you didn't mention it again. That's conversation context in action!

## Step 6: Clean Up
Always delete agents when done to keep your Foundry project clean and avoid unnecessary resource usage.

In [ ]:
project_client.agents.delete(agent.name)
print(f"✅ Agent '{agent.name}' deleted")

## ✅ Lab 1 Complete!

### Key Takeaways
| Concept | What You Learned |
|---------|------------------|
| `PromptAgentDefinition` | Define agent behavior with model + instructions |
| `create_version()` | Register versioned agents in Foundry |
| `agent_reference` | Route requests to your specific agent |
| `conversations.create()` | Enable multi-turn stateful conversations |
| Agent cleanup | Delete agents when no longer needed |

---
**Next →** [Lab 2: Prepare Production Data](lab2_generate_data.ipynb) — Generate a realistic 500-record insurance claims dataset